<a href="https://colab.research.google.com/github/Luffy-13/Fintech-news/blob/main/Fintech_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import requests
import feedparser
from datetime import datetime, timedelta

# ------------ CONFIG --------------------
NEWS_API_KEY = '55529eb912da4182909ddc6d56e7bf03'  # Your NewsAPI Key
TELEGRAM_BOT_TOKEN = '8150622631:AAGyZz1lbexgF9C1eNx8S-xeQm8kbd-4vfs'  # Your Telegram Bot Token
CHAT_ID = '1560336570'  # Your actual Telegram chat ID

# Expanded keywords for Indian economy
KEYWORDS = [
    'Indian economy', 'GDP', 'inflation', 'stock market', 'monetary policy', 'economic growth',
    'business growth', 'PM Modi', 'trade balance', 'government policies', 'budget',
    'global economy', 'economic reforms', 'business investments', 'corporate earnings',
    'RBI', 'market trends', 'exports', 'financial crisis'
]

# Expanded RSS feeds for economic and business news
RSS_FEEDS = [
    'https://economictimes.indiatimes.com/rssfeedsdefault.cms',
    'https://www.business-standard.com/rss/india-others.xml',
    'https://indianexpress.com/feed/',
    'https://www.thehindubusinessline.com/rssfeed/',
    'https://www.livemint.com/rss/',
    'https://www.financialexpress.com/feed/'
]

# ------------ FUNCTIONS ------------------

def fetch_newsapi():
    url = (
        f'https://newsapi.org/v2/everything?'
        f'q=Indian economy&from={(datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")}'  # 1 week range
        f'&language=en&sortBy=publishedAt&apiKey={NEWS_API_KEY}'
    )
    response = requests.get(url)
    articles = response.json().get('articles', [])
    return [
        {
            'title': a['title'],
            'url': a['url'],
            'source': a['source']['name'],
            'publishedAt': a['publishedAt']
        }
        for a in articles
    ]

def fetch_rss():
    all_entries = []
    for feed_url in RSS_FEEDS:
        feed = feedparser.parse(feed_url)
        for entry in feed.entries:
            all_entries.append({
                'title': entry.title,
                'url': entry.link,
                'source': feed.feed.get('title', 'Unknown'),
                'publishedAt': entry.get('published', 'N/A')
            })
    return all_entries

def filter_articles(articles):
    filtered = []
    for article in articles:
        if any(kw.lower() in article['title'].lower() for kw in KEYWORDS):
            filtered.append(article)
    return filtered

def format_message(articles):
    message = f"\U0001F4F0 *Daily Economic Digest* ({datetime.now().strftime('%Y-%m-%d')})\n\n"
    for idx, art in enumerate(articles, 1):
        message += f"*{idx}. {art['title']}*\n_Source_: {art['source']}\n[Read more]({art['url']})\n\n"
    return message

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        'chat_id': CHAT_ID,
        'text': message,
        'parse_mode': 'Markdown',
        'disable_web_page_preview': True
    }
    r = requests.post(url, data=payload)
    print(r.json())  # Debugging line to see API response
    return r.json()

# ------------ LOGGING ---------------
import sys
sys.stdout = open('output.log', 'w')  # Redirect all print statements to a log file

# ------------- RUN ------------------

# Fetch news
print("Fetching economy-related news...")
newsapi_articles = fetch_newsapi()
rss_articles = fetch_rss()

# Combine and filter
all_articles = newsapi_articles + rss_articles
filtered_articles = filter_articles(all_articles)

# Format and send
if filtered_articles:
    message = format_message(filtered_articles[:3])  # send top 5 only
    send_telegram(message)
    print("✅ Sent economic news to Telegram!")
else:
    send_telegram("No economic updates found today.")
    print("ℹ️ No updates found today.")
